# Baby Step 8 — Controlled Outreach Governance and Synthetic Market Testing

This notebook tests recipient verification, conflict refresh, disclosure tiers, contact logging and response capture using fictional recipients and reserved `.invalid` domains.

> **Safety:** it cannot send messages and does not authorize real outreach.


## Control question

Can the operating system simulate an outreach workflow while proving that identity, conflicts, disclosure, logging and human authorization remain enforced?


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os
import pandas as pd
import numpy as np
try:
    from IPython.display import display
except ImportError:
    display = print

VAULT_NAME = "Alejandro-Reynoso-Investment-Banking-Vault"
env = os.environ.get("VAULT_PATH")
candidates = ([Path(env)] if env else []) + [Path.cwd()/VAULT_NAME, Path("/content/drive/MyDrive")/VAULT_NAME, Path("/workspace/scratch/9ba1ff46ede5")/VAULT_NAME]
VAULT = next((p for p in candidates if p.exists()), None)
if VAULT is None: raise FileNotFoundError("Set VAULT_PATH or place the vault in the current directory or Google Drive.")
SIMULATION_MODE = True
EXTERNAL_SEND_ENABLED = False
print("Vault:", VAULT)
print("Simulation mode:", SIMULATION_MODE, "External send:", EXTERNAL_SEND_ENABLED)


## 1. Validate governed inputs


In [ ]:
required = [
    VAULT/"Data/loop_007_shortlist.csv", VAULT/"Data/loop_008_recipient_verification.csv",
    VAULT/"Data/loop_008_disclosure_matrix.csv", VAULT/"Data/loop_008_synthetic_market_test.csv",
    VAULT/"Data/loop_008_control_checks.csv", VAULT/"Data/claim_register.csv", VAULT/"Data/source_registry.csv",
]
missing=[str(p) for p in required if not p.exists()]
assert not missing, missing
print("Validated",len(required),"inputs")


## 2. Load the short list, recipients and controls


In [ ]:
shortlist=pd.read_csv(required[0]); recipients=pd.read_csv(required[1]); disclosure=pd.read_csv(required[2]); market=pd.read_csv(required[3]); controls=pd.read_csv(required[4]); claims=pd.read_csv(required[5]); sources=pd.read_csv(required[6])
assert len(shortlist)==9 and len(recipients)==9
assert len(claims)==30 and len(sources)==15
display(recipients[["counterparty_name","recipient_name","role","conflict_refresh","simulation_status"]])


## 3. Verify identity, authority and safe synthetic domains


In [ ]:
assert recipients["recipient_id"].is_unique
assert recipients["identity_verified"].all()
assert recipients["authority_verified"].all()
assert recipients["synthetic_domain"].str.endswith(".invalid").all()
assert not recipients["synthetic_domain"].str.contains("@",regex=False).any()
print("All recipient and reserved-domain controls pass.")


## 4. Refresh conflicts and enforce Wave 2 blocks


In [ ]:
wave1=recipients.query('simulation_status == "Wave 1 simulation"')
blocked=recipients.query('simulation_status.str.startswith("Blocked")',engine="python")
assert len(wave1)==6 and len(blocked)==3
assert (wave1["conflict_refresh"]=="Clear").all()
assert (blocked["conflict_refresh"]=="Conditional").all()
display(blocked[["counterparty_name","conflict_refresh","simulation_status"]])


## 5. Validate the disclosure sequence


In [ ]:
assert disclosure["tier"].tolist()==["Tier 0","Tier 1","Tier 2","Tier 3","Tier 4"]
assert not disclosure["external_send_allowed"].any()
assert (wave1["maximum_disclosure_tier"]=="Tier 1").all()
print("Disclosure is ordered and external sending is disabled at every tier.")
display(disclosure[["tier","name","prerequisites","external_send_allowed"]])


## 6. Inspect the six simulated contacts


In [ ]:
assert len(market)==6
assert market["simulation_id"].is_unique
assert (market["protocol_status"]=="PASS").all()
assert (market["disclosure_tier"]=="Tier 1").all()
assert (market["transmission_status"]=="No external transmission").all()
display(market[["counterparty_name","response_class","synthetic_feedback"]])


## 7. Normalize the synthetic response set


In [ ]:
response_counts=market["response_class"].value_counts().reindex(["Positive","Conditional","Hold"]).fillna(0).astype(int)
assert response_counts.to_dict()=={"Positive":4,"Conditional":1,"Hold":1}
print(response_counts.to_dict())
ax=response_counts.plot(kind="bar",color=["#2A7F8E","#D7863B","#687786"],title="Synthetic market-test responses",ylabel="Count",rot=0)
ax.spines[["top","right"]].set_visible(False)


## 8. Run the twelve protocol-control tests


In [ ]:
assert len(controls)==12
assert (controls["status"]=="PASS").all()
display(controls)
print("Control pass rate: 12 / 12")


## 9. Prove that the notebook has no send path


In [ ]:
safety = {
    "simulation_mode": SIMULATION_MODE,
    "external_send_disabled": not EXTERNAL_SEND_ENABLED,
    "reserved_domains_only": recipients["synthetic_domain"].str.endswith(".invalid").all(),
    "tier_1_only": (market["disclosure_tier"]=="Tier 1").all(),
    "no_transmission": (market["transmission_status"]=="No external transmission").all(),
}
assert all(bool(v) for v in safety.values())
print({k:bool(v) for k,v in safety.items()})


## 10. Run the Step 8 decision engine


In [ ]:
decision_checks={
    "six_wave1_simulations": len(wave1)==6 and len(market)==6,
    "three_wave2_blocked": len(blocked)==3,
    "all_controls_pass": (controls["status"]=="PASS").all(),
    "response_set_complete": response_counts.sum()==6,
    "registers_advanced": len(sources)==15 and len(claims)==30,
    "no_external_action": not EXTERNAL_SEND_ENABLED,
}
assert all(bool(v) for v in decision_checks.values())
recommendation="APPROVE SYNTHETIC PROTOCOL — NO REAL OUTREACH"
print({k:bool(v) for k,v in decision_checks.items()})
print(recommendation)


## 11. Optional dry-run manifest


In [ ]:
manifest={
    "generated_at_utc":datetime.now(timezone.utc).isoformat(),"loop":"008","recommendation":recommendation,
    "responses":{k:int(v) for k,v in response_counts.items()},"controls_passed":int((controls["status"]=="PASS").sum()),
    "real_contacts":0,"external_send_enabled":False,
}
print(json.dumps(manifest,indent=2))


## 12. Define the Baby Step 9 input contract


In [ ]:
next_contract=pd.DataFrame([
    ["New information ingestion","Company, market and environmental events"],
    ["Freshness engine","Identify stale or superseded claims"],
    ["Affected subgraph","Recalculate only exposed companies and opportunities"],
    ["Rebalance gate","Preserve baseline and write a new scenario state"],
],columns=["Component","Required result"])
display(next_contract)


## What Baby Step 8 demonstrates

The operating system can model the movement from selection to communication without silently creating authority:

**Short list → recipient verification → conflict refresh → disclosure tier → logged simulation → normalized response → human gate.**
